In [ ]:
import os
os.environ["RAY_DEDUP_LOGS"] = "0"
os.environ["RAY_DISABLE_METRICS"] = "1"
os.environ["RAY_DISABLE_METRICS_EXPORT"] = "1"

In [2]:

import polars as pl
from sklearn.metrics import accuracy_score

from autotsc.utils import load_dataset
from time import perf_counter

import numpy as np
import polars as pl
import ray
from aeon.classification.dictionary_based import REDCOMETS
from aeon.classification.interval_based import SupervisedTimeSeriesForest

from aeon.classification.base import BaseClassifier
from aeon.classification.convolution_based import (
    MiniRocketClassifier,
    MultiRocketClassifier,
    RocketClassifier,
)
from aeon.classification.dummy import DummyClassifier
from aeon.classification.feature_based import (
    Catch22Classifier,
    SummaryClassifier,
)
from aeon.classification.interval_based import (
    QUANTClassifier,
)
from aeon.classification.sklearn import SklearnClassifierWrapper
from aeon.transformations.collection import DownsampleTransformer
from sklearn.base import clone
from sklearn.ensemble import (
    ExtraTreesClassifier,
    HistGradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.linear_model import RidgeClassifierCV
from sklearn.metrics import accuracy_score
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler
import types
from autotsc.models import RidgeClassifierCVWithProba
from aeon.classification.convolution_based import HydraClassifier
from autotsc import transformers, utils
from aeon.classification.dictionary_based import IndividualBOSS
from aeon.classification.interval_based import (
    RSTSF,
    CanonicalIntervalForestClassifier,
    DrCIFClassifier,
    QUANTClassifier,
    RandomIntervalSpectralEnsembleClassifier,
    SupervisedTimeSeriesForest,
    TimeSeriesForestClassifier,
)
from aeon.classification.interval_based import RSTSF
from aeon.classification.dictionary_based import WEASEL_V2
from aeon.classification.dictionary_based import ContractableBOSS
from aeon.classification.hybrid import RISTClassifier
from aeon.pipeline import make_pipeline as aeon_make_pipeline
import hvplot.polars
import holoviews as hv
from aeon.classification.convolution_based import MultiRocketHydraClassifier

2025-11-27 10:28:16.636007: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [3]:
X_train, y_train, X_test, y_test = load_dataset("CricketZ")
#X_train, y_train, X_test, y_test = load_dataset("MelbournePedestrian")
#X_train, y_train, X_test, y_test = load_dataset("Worms")
# X_train, y_train, X_test, y_test = load_dataset("PhalangesOutlinesCorrect")
# X_train, y_train, X_test, y_test = load_dataset("Lightning2")
# X_train, y_train, X_test, y_test = load_dataset("CricketY")
#X_train, y_train, X_test, y_test = load_dataset("OSULeaf")
#X_train, y_train, X_test, y_test = load_dataset("Wine")
#X_train, y_train, X_test, y_test = load_dataset("FiftyWords")
#X_train, y_train, X_test, y_test = load_dataset("Haptics")

	
#X_train, y_train, X_test, y_test = load_dataset("InlineSkate")
X_train, y_train, X_test, y_test = load_dataset("SemgHandSubjectCh2")

In [4]:
@ray.remote(num_cpus=1)
def ray_run_model_on_fold(model_clone, X, y, train_idx, valid_idx):
    X_train, y_train = X[train_idx], y[train_idx]
    X_valid, y_valid = X[valid_idx], y[valid_idx]
    start_time = perf_counter()
    model_clone.fit(X_train, y_train)
    fit_time = perf_counter() - start_time
    proba = model_clone.predict_proba(X_valid)
    val_probs = list(zip(valid_idx, proba))
    # print(f"Train model {model_clone.__class__.__name__} on fold done in {fit_time:.2f}s")
    return model_clone, val_probs, fit_time

@ray.remote(num_cpus=1)
def ray_run_predict_proba(model, X):
    start_time = perf_counter()
    proba = model.predict_proba(X)
    pred_time = perf_counter() - start_time
    # print(f"Predict proba with model {model.__class__.__name__} done in {pred_time:.2f}s")
    return proba

@ray.remote(num_cpus=0, resources={"meta": 1})
def ray_run_fit_predict_proba_wrapper(model_id, wrapper, X, y, folds):
    start_fp = perf_counter()
    result = wrapper.fit_predict_proba(X, y, cv_splits=folds)
    total_fp_time = perf_counter() - start_fp
    print(f"Completed fit_predict_proba in Ray task in {total_fp_time:.2f}s")
    return model_id, wrapper, result

class RayCrossValidationWrapper(BaseClassifier):
    def __init__(self, model):
        super().__init__()
        self.model = model
        self.trained_models_ = []
        self.fit_time_ = []
        self.fit_time_mean_ = None

    def _fit(self, X, y):
        raise NotImplementedError()

    def _predict(self, X):
        tasks = [ray_run_predict_proba.remote(model, X) for model in self.trained_models_]
        predictions = ray.get(tasks)
        avg_proba = np.mean(predictions, axis=0)
        predicted_indices = np.argmax(avg_proba, axis=1)
        return self.classes_[predicted_indices]

    def _fit_predict_proba(self, X, y, cv_splits):
        n_classes = len(np.unique(y))
        oof_proba = np.zeros((len(y), n_classes))

        fold_tasks = []
        for train_idx, val_idx in cv_splits:
            model_clone = clone(self.model)
            task = ray_run_model_on_fold.remote(model_clone, X, y, train_idx, val_idx)
            fold_tasks.append(task)

        results = ray.get(fold_tasks)
        for model, proba, fit_time in results:
            self.trained_models_.append(model)
            for idx, p in proba:
                oof_proba[idx] = p
            self.fit_time_.append(fit_time)
        self.fit_time_mean_ = np.mean(self.fit_time_)
        return oof_proba

    def fit_predict_proba(self, X, y, cv_splits):
        X, y, single_class = self._fit_setup(X, y)
        y_proba = self._fit_predict_proba(X, y, cv_splits)
        self.is_fitted = True
        return y_proba

#with utils.ray_init_or_reuse(num_cpus=24):
#    m = RayCrossValidationWrapper(Catch22Classifier(n_jobs=1))
#    folds = utils.get_folds(X_train, y_train, n_splits=5)
#    pred_proba = m.fit_predict_proba(X_train, y_train, folds)
#    print(pred_proba.shape)
#

In [5]:
X_train.shape

(450, 1, 1500)

In [6]:
class Ensemble:
    def __init__(self, model):
        self.model_ = model
        self.models = []

    def fit_predict_proba_cv(self, X, y, cv_splits):
        proba_predictions = []
        for train_idx, val_idx in cv_splits:
            model_clone = clone(self.model_)
            model_clone.fit(X[train_idx], y[train_idx])
            self.models.append(model_clone)
            y_proba = model_clone.predict_proba(X[val_idx])
            proba_predictions.extend(zip(val_idx, y_proba))
        proba_predictions = sorted(proba_predictions)
        return np.array([proba for idx, proba in proba_predictions])
    
    def predict(self, X):
        predictions = np.array([model.predict_proba(X) for model in self.models])
        avg_proba = predictions.mean(axis=0)
        predicted_indices = np.argmax(avg_proba, axis=1)
        return self.models[0].classes_[predicted_indices]
    
    def __repr__(self):
        return f'{self.__class__.__name__}({self.model_})'

In [7]:
class EnsambleWeights:
    def __init__(self):
        pass

In [8]:
class AutoTSCModel(BaseClassifier):

    # TODO: change capability tags
    _tags = {
        "capability:multivariate": True,
        "capability:train_estimate": True,
        "capability:contractable": True,
        "capability:multithreading": True,
        "algorithm_type": "convolution",
    }

    def __init__(self):
        self.models_ = {}
        self.meta_models_ = {}
        self.summary_ = []
        self.oof_predictions_ = None
        self.verbose = 1
        self.n_jobs = -1
        self.n_gpus = -1
        super().__init__()

    def get_default_ray_models(self, random_seed):
        
        return [   
            ('tab-ridge', RayCrossValidationWrapper(
                SklearnClassifierWrapper(RidgeClassifierCVWithProba(alphas=np.logspace(-3, 3, 10)))
            )),
            ('tab-rf', RayCrossValidationWrapper(
                SklearnClassifierWrapper(RandomForestClassifier(n_jobs=-1, n_estimators=500))
            )),
            ('dif-roc', RayCrossValidationWrapper(aeon_make_pipeline(
                transformers.Difference(),
                RocketClassifier(n_jobs=1, estimator=RidgeClassifierCVWithProba(alphas=np.logspace(-3, 3, 10)), random_state=random_seed)
            ))),
            ('cs-roc', RayCrossValidationWrapper(aeon_make_pipeline(
                transformers.CumSum(),
                MultiRocketClassifier(n_jobs=1, estimator=RidgeClassifierCVWithProba(alphas=np.logspace(-3, 3, 10)), random_state=random_seed)
            ))),
            ('c22', RayCrossValidationWrapper(Catch22Classifier(n_jobs=1))),
            ('roc', RayCrossValidationWrapper(RocketClassifier(n_jobs=1, estimator=RidgeClassifierCVWithProba(alphas=np.logspace(-3, 3, 10)), random_state=random_seed))),
            ('minroc', RayCrossValidationWrapper(MiniRocketClassifier(n_jobs=1, estimator=RidgeClassifierCVWithProba(alphas=np.logspace(-3, 3, 10)), random_state=random_seed))),
            ('quant', RayCrossValidationWrapper(
                aeon_make_pipeline(
                    transformers.CumSum(),
                    QUANTClassifier(random_state=random_seed)
                )
            )),
            ('dif-quant', RayCrossValidationWrapper(
                aeon_make_pipeline(
                    transformers.Difference(),
                    QUANTClassifier(random_state=random_seed)
                )
            )),
            ('cs-quant', RayCrossValidationWrapper(QUANTClassifier(random_state=random_seed))),

            ('hydra', RayCrossValidationWrapper(HydraClassifier(n_jobs=1, random_state=random_seed))),
            ('mulroc', RayCrossValidationWrapper(MultiRocketClassifier(n_jobs=1, estimator=RidgeClassifierCVWithProba(alphas=np.logspace(-3, 3, 10)), random_state=random_seed))),
            ('weasel', RayCrossValidationWrapper(WEASEL_V2(n_jobs=1, max_feature_count=10000))),
            ('tsf', RayCrossValidationWrapper(SupervisedTimeSeriesForest(n_jobs=1, n_estimators=20))),
            ('rstsf', RayCrossValidationWrapper(RSTSF(n_jobs=1, n_estimators=20))),
            ('summry', RayCrossValidationWrapper(SummaryClassifier(n_jobs=1))),
            ('rochydra', RayCrossValidationWrapper(MultiRocketHydraClassifier(n_jobs=1, random_state=random_seed+1))),
            ('cboss', RayCrossValidationWrapper(ContractableBOSS(n_jobs=1, time_limit_in_minutes=1.0))),
            ('drcif', RayCrossValidationWrapper(DrCIFClassifier(n_jobs=1, random_state=random_seed, time_limit_in_minutes=1.0))),
            # RayCrossValidationWrapper(RISTClassifier(n_jobs=1)),
            #RayCrossValidationWrapper(RDSTClassifier(n_jobs=-1)),
        ]
        #model6 = CVWrapper(
        #    IndividualBOSS(n_jobs=-1, random_state=42)
        #)
        #model7 = CVWrapper(
        #    DrCIFClassifier(n_jobs=-1, random_state=42, n_estimators=5)
        #)
        #model8 = CVWrapper(
        #    REDCOMETS(n_jobs=-1, n_trees=50)
        #)
        #model13 = CVWrapper(
        #    ContractableBOSS(n_jobs=-1, time_limit_in_minutes=1)
        #)

    def get_default_metamodels(self):
        from sklearn.decomposition import PCA
        model1 = ('m-ridgecv', Ensemble(RidgeClassifierCVWithProba(alphas=np.logspace(-3, 3, 10))))
        model2 = ('m-rf', Ensemble(RandomForestClassifier(n_jobs=-1, n_estimators=500)))
        model3 = ('m-hgb', Ensemble(HistGradientBoostingClassifier()))
        model4 = ('m-et', Ensemble(ExtraTreesClassifier(n_jobs=-1, n_estimators=500)))
        model5 = ('m-rf-pruned', Ensemble(RandomForestClassifier(n_jobs=-1, n_estimators=500, ccp_alpha=0.01)))
        model6 = ('m-pca-rf', Ensemble(
            make_pipeline(
                StandardScaler(),
                PCA(n_components=0.95),
                RandomForestClassifier(n_jobs=-1, n_estimators=500, ccp_alpha=0.01)
            )
        ))
        from sklearn.svm import LinearSVC
        from sklearn.svm import SVC
        from sklearn.linear_model import LogisticRegressionCV
        # train linear svm
        model7 = ('m-svm', Ensemble(
            SVC(kernel='linear', probability=True)
        ))
        model8 = ('m-log', Ensemble(
            LogisticRegressionCV(cv=5, n_jobs=-1)
        )) # 
        # add catboost
        #from catboost import CatBoostClassifier
        #model6 = Ensemble(
        #    CatBoostClassifier()
        #)
        return [model1, 
                #model2, 
                #model3, 
                #model4, 
                #model5, 
                #model6,
                model7,
                model8
        ]

    def _fit(self, X, y):
        self.cpus_available_, self.cpus_to_use_, self.gpus_available_, self.gpus_to_use_ = (
            utils.get_resource_config(self.n_jobs, self.n_gpus)
        )
        random_seed = np.random.randint(0, 10000)
        n_folds = 20
        if self.verbose > 0:
            utils.print_fit_start_info(
                X,
                y,
                self.cpus_to_use_,
                self.cpus_available_,
                self.gpus_to_use_,
                self.gpus_available_,
                random_seed,
                n_folds=n_folds,
            )
        folds = utils.get_folds(X, y, n_splits=n_folds)
        #default_models = self.get_default_models()
        default_ray_models = self.get_default_ray_models(random_seed)

        ts = []
        # create X and y reference
        X_ref = ray.put(X)
        y_ref = ray.put(y)
        for model_id, model in default_ray_models:
            t = ray_run_fit_predict_proba_wrapper.remote(model_id, model, X_ref, y_ref, folds)
            ts.append(t)

        ray_results = ray.get(ts)
        for model_id, model, pred in ray_results:
            self.models_[model_id] = model
            pred_max = np.argmax(pred, axis=1)
            acc = accuracy_score(y, model.classes_[pred_max])
            self.summary_.append({
                "model_id": model_id,
                "model": repr(model).replace("\n", "").replace(" ", ""),
                "validation_accuracy": acc,
                "stacking_level": 0,
                "train_time": model.fit_time_mean_,
            })

            if self.verbose > 0:
                print(f"Trained base model {model_id}, OOF accuracy: {acc:.4f} in {model.fit_time_mean_:.2f}s")

            columns = [f'model_{model_id}__class_{l}' for l in list(model.classes_)]
            if self.oof_predictions_ is None:
                self.oof_predictions_ = pl.DataFrame(pred, schema=columns)
            else:
                df_pred = pl.DataFrame(pred, schema=columns)
                self.oof_predictions_ = pl.concat([self.oof_predictions_, df_pred], how='horizontal')

        default_metamodels = self.get_default_metamodels()
        for model_id, model in default_metamodels:
            start_time = perf_counter()
            pred = model.fit_predict_proba_cv(self.oof_predictions_.to_numpy(), y, folds)
            endtime = perf_counter()
            pred_max = np.argmax(pred, axis=1)
            acc = accuracy_score(y, model.models[0].classes_[pred_max])
            self.meta_models_[model_id] = model
            self.summary_.append({
                "model_id": model_id, 
                "model": repr(model).replace("\n", "").replace(" ", ""),
                "validation_accuracy": acc,
                "stacking_level": 1,
            })
            if self.verbose > 0:
                print(f"Trained meta model {model_id}, OOF accuracy: {acc:.4f} in {endtime - start_time:.2f}s")

        return self
    
    def summary(self):
        return pl.DataFrame(self.summary_)
    
    def predict_per_model(self, X):
        "make predictions for each model in the ensemble"
        all_preds = {}
        oof_predictions_ = None
        for model_id, model in self.models_.items():
            pred_probs = model.predict_proba(X)
            columns = [f'model_{model_id}__class_{l}' for l in list(model.classes_)]
            if oof_predictions_ is None:
                oof_predictions_ = pl.DataFrame(pred_probs, schema=columns)
            else:
                df_pred = pl.DataFrame(pred_probs, schema=columns)
                oof_predictions_ = pl.concat([oof_predictions_, df_pred], how='horizontal')
            pred = np.argmax(pred_probs, axis=1)
            pred_labels = model.classes_[pred]
            all_preds[model_id] = pred_labels

        for model_id, model in self.meta_models_.items():
            pred = model.predict(oof_predictions_.to_numpy())
            all_preds[model_id] = pred

        return all_preds

    def _predict(self, X):
        m = self.models_[list(self.models_.keys())[0]]
        return m._predict(X)

#folds = utils.get_folds(X_train, y_train, n_splits=10)

In [ ]:
with utils.ray_init_or_reuse(num_cpus=24, resources={"meta": 100}, ignore_reinit_error=True):
    model = AutoTSCModel()
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    accuracy_score(y_test, pred)

2025-11-27 10:28:22,375	INFO worker.py:2023 -- Started a local Ray instance.
/home/gasper_p/workspace/repos/AutoTSC/.venv/lib/python3.12/site-packages/ray/_private/worker.py:2062: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


|------------------------|
| Number of samples: 450 |
| Number of channels: 1  |
| Length of series: 1500 |
| Number of classes: 5   |
| CPUs: 24/24            |
| GPUs: 2/2              |
| Random seed: 2023      |
| Number of folds: 20    |
|------------------------|


(ray_run_fit_predict_proba_wrapper pid=2642208) 2025-11-27 10:28:29.904033: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
(ray_run_fit_predict_proba_wrapper pid=2642208) To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
(ray_run_fit_predict_proba_wrapper pid=2642215) 2025-11-27 10:28:30.255020: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
(ray_run_fit_predict_proba_wrapper pid=2642215) To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
(ray_run_fit_predict_proba_wrapper pid=2642219) 2025-11-27 10:28:31.162858: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CP

(ray_run_fit_predict_proba_wrapper pid=2642201) Completed fit_predict_proba in Ray task in 9.02s


(pid=gcs_server) [2025-11-27 10:28:50,991 E 2641963 2641963] (gcs_server) gcs_server.cc:303: Failed to establish connection to the event+metrics exporter agent. Events and metrics will not be exported. Exporter agent status: RpcError: Running out of retries to initialize the metrics agent. rpc_code: 14
(raylet) [2025-11-27 10:28:52,304 E 2642122 2642122] (raylet) main.cc:979: Failed to establish connection to the metrics exporter agent. Metrics will not be exported. Exporter agent status: RpcError: Running out of retries to initialize the metrics agent. rpc_code: 14
(ray_run_fit_predict_proba_wrapper pid=2642204) [2025-11-27 10:28:54,391 E 2642204 2642714] core_worker_process.cc:837: Failed to establish connection to the metrics exporter agent. Metrics will not be exported. Exporter agent status: RpcError: Running out of retries to initialize the metrics agent. rpc_code: 14
(ray_run_fit_predict_proba_wrapper pid=2642202) [2025-11-27 10:28:54,389 E 2642202 2642673] core_worker_process.c

(ray_run_fit_predict_proba_wrapper pid=2642212) Completed fit_predict_proba in Ray task in 41.07s
(ray_run_fit_predict_proba_wrapper pid=2642200) Completed fit_predict_proba in Ray task in 49.72s
(ray_run_fit_predict_proba_wrapper pid=2642206) Completed fit_predict_proba in Ray task in 52.93s


(ray_run_model_on_fold pid=2643705) /home/gasper_p/workspace/repos/AutoTSC/.venv/lib/python3.12/site-packages/joblib/parallel.py:1383: UserWarning: The backend class 'SequentialBackend' does not support timeout. You have set 'timeout=99999' in Parallel but the 'timeout' parameter will not be used.
(ray_run_model_on_fold pid=2643705)   warnings.warn(
(ray_run_model_on_fold pid=2643671) /home/gasper_p/workspace/repos/AutoTSC/.venv/lib/python3.12/site-packages/joblib/parallel.py:1383: UserWarning: The backend class 'SequentialBackend' does not support timeout. You have set 'timeout=99999' in Parallel but the 'timeout' parameter will not be used.
(ray_run_model_on_fold pid=2643671)   warnings.warn(
(ray_run_model_on_fold pid=2643721) /home/gasper_p/workspace/repos/AutoTSC/.venv/lib/python3.12/site-packages/joblib/parallel.py:1383: UserWarning: The backend class 'SequentialBackend' does not support timeout. You have set 'timeout=99999' in Parallel but the 'timeout' parameter will not be use

(ray_run_fit_predict_proba_wrapper pid=2642203) Completed fit_predict_proba in Ray task in 329.82s


In [ ]:
summary = model.summary()
summary

In [ ]:
test_accs = []
preds = model.predict_per_model(X_test)
for m, p in preds.items():
    acc = accuracy_score(y_test, p)
    test_accs.append({
        "model_id": m,
        "test_accuracy": acc,
    })
test_accs = pl.DataFrame(test_accs)
summary = summary.join(test_accs, on="model_id")

In [ ]:
pl.Config.set_fmt_str_lengths(500)
for r in summary.sort("validation_accuracy").iter_rows():
    print(r)

In [ ]:
# Create the scatter plot
scatter = summary.hvplot.scatter(
    x="validation_accuracy",
    y="test_accuracy",
    c="stacking_level",
    hover_cols=["model_id", "validation_accuracy", "test_accuracy", "stacking_level"],
    cmap="viridis",
    size=80,
    alpha=0.7,
    title="Validation vs Test Accuracy per Model",
    xlabel="Validation Accuracy",
    ylabel="Test Accuracy",
    width=700,
    height=600,
    grid=True  # Add grid here
)

# Add text labels for model_id
labels = hv.Labels(
    summary.select(["validation_accuracy", "test_accuracy", "model_id"]).to_pandas(),
    kdims=["validation_accuracy", "test_accuracy"],
    vdims=["model_id"]
).opts(
    text_font_size="8pt",
    text_align="left",
    text_baseline="bottom",
    xoffset=0.0005,  # slight offset so text doesn't overlap point
    yoffset=0.0005
)

# Add diagonal line
min_val = min(summary["validation_accuracy"].min(), summary["test_accuracy"].min())
max_val = max(summary["validation_accuracy"].max(), summary["test_accuracy"].max())
diagonal = hv.Curve([(min_val, min_val), (max_val, max_val)]).opts(
    color="gray", 
    line_dash="dashed",
    line_width=1
)

# Combine all
plot = scatter * labels * diagonal
plot